**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 01 - PDF text extraction

Reading the RBI Handbook of Statistics on Indian States 2024-25 and pulling the raw
text out of the PDF. Later notebooks locate and parse the tables from this text.

The PDF in data/raw is never edited. Extracted text goes to data/processed.

## Setup
Install pdfplumber, the library used to read the PDF.

In [1]:
!pip install pdfplumber


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Check the folder
Print the current working directory and the files in data/raw.

In [3]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("Files in data/raw:", list(Path("../data/raw").glob("*")))

Current working directory: D:\india-state-competitiveness-index
Files in data/raw: []


### Check the path again
List data/raw with a relative path to confirm where the notebook is running from.

In [4]:
from pathlib import Path
print(list(Path("data/raw").glob("*")))

[WindowsPath('data/raw/. Startup India  DPIIT (State-wise Startup Data).csv'), WindowsPath('data/raw/Annual Report of Ministry of MSME (2025–26).pdf'), WindowsPath('data/raw/GSDP.xlsx'), WindowsPath('data/raw/HANDBOOK OF STATISTICS ON THE INDIAN ECONOMY 2023-24 RBI.pdf'), WindowsPath('data/raw/Monthly Infrastructure Performance Review Report – JUNE 2025.pdf'), WindowsPath('data/raw/PLFS_changes_in_2025_Final.pdf'), WindowsPath('data/raw/RBI Handbook of Statistics on Indian States 2024-25.pdf'), WindowsPath('data/raw/README.md'), WindowsPath('data/raw/Startup IndiaDPIIT (State-wise Startup Data)PDF.pdf'), WindowsPath('data/raw/Sustainable Development Goals – National Indicator Framework Progress Report 2026.pdf')]


### Open the PDF
Open the handbook, count its pages, and print sample text from one page.

In [5]:
import pdfplumber
from pathlib import Path

pdf_path = Path("data/raw/RBI Handbook of Statistics on Indian States 2024-25.pdf")

with pdfplumber.open(pdf_path) as pdf:
    print(len(pdf.pages), "pages")
    text = pdf.pages[9].extract_text() or ""

print(text[:800])

472 pages
77 State-wise Area of Non-Foodgrains - Cotton (Lint) 2004-05 to 2024-25 181
78 State-wise Area of Non-Foodgrains - Sugarcane 2004-05 to 2024-25 183
79 State-wise Area of Non-Foodgrains - Raw Jute & Mesta 2004-05 to 2024-25 185
80 State-wise Estimates of Yield - Rice 2004-05 to 2024-25 187
81 State-wise Estimates of Yield - Wheat 2004-05 to 2024-25 189
82 State-wise Estimates of Yield - Coarse Cereals 2004-05 to 2024-25 191
83 State-wise Estimates of Yield - Pulses 2004-05 to 2024-25 193
84 State-wise Estimates of Yield - Total Foodgrains 2004-05 to 2024-25 195
85 State-wise Estimates of Yield - Oilseeds 2004-05 to 2024-25 197
86 State-wise Estimates of Yield - Cotton (Lint) 2004-05 to 2024-25 199
87 State-wise Estimates of Yield - Sugarcane 2004-05 to 2024-25 201
88 State-wise Estimates of


### Save all the text
Extract text from every page and save it to one file in data/processed. The PDF is not read again after this.

In [6]:
out_path = Path("data/processed/rbi_states_handbook_2024_25.txt")

with pdfplumber.open(pdf_path) as pdf:
    pages = [page.extract_text() or "" for page in pdf.pages]

full_text = "\n".join(pages)
out_path.write_text(full_text, encoding="utf-8")

print("Saved", len(full_text), "characters to", out_path)
print("Pages processed:", len(pages))

Saved 1174062 characters to data\processed\rbi_states_handbook_2024_25.txt
Pages processed: 472


### Find the table headings
Scan the saved text for lines that start with TABLE.

In [7]:
full_text = Path("data/processed/rbi_states_handbook_2024_25.txt").read_text(encoding="utf-8")

count = 0
for line in full_text.splitlines():
    if line.strip().upper().startswith("TABLE "):
        print(repr(line))
        count += 1
        if count >= 20:
            break

'Table Page'
'Table Title Time Period'
'TABLE 1: STATE-WISE GROSS ENROLMENT RATIO- 2022-23'
'TABLE 1: STATE-WISE GROSS ENROLMENT RATIO- 2023-24'
'TABLE 1: STATE-WISE GROSS ENROLMENT RATIO- 2024-25'
'TABLE 2: STATE-WISE BIRTH RATE'
'TABLE 2: STATE-WISE BIRTH RATE (Concld.)'
'TABLE 3: STATE-WISE DEATH RATE'
'TABLE 3: STATE-WISE DEATH RATE (Concld.)'
'TABLE 4: STATE-WISE INFANT MORTALITY RATE'
'TABLE 4: STATE-WISE INFANT MORTALITY RATE (Concld.)'
'TABLE 5: STATE-WISE MATERNAL MORTALITY RATIO'
'TABLE 6: STATE-WISE TOTAL FERTILITY RATE'
'TABLE 6: STATE-WISE TOTAL FERTILITY RATE (Concld.)'
'TABLE 7: STATE-WISE LIFE EXPECTANCY'
'TABLE 7: STATE-WISE LIFE EXPECTANCY (Contd.)'
'TABLE 7: STATE-WISE LIFE EXPECTANCY (Contd.)'
'TABLE 7: STATE-WISE LIFE EXPECTANCY (Contd.)'
'TABLE 7: STATE-WISE LIFE EXPECTANCY (Contd.)'
'TABLE 7: STATE-WISE LIFE EXPECTANCY (Contd.)'


## Locating and parsing the simple tables
locate_table pulls out the text block for one table number. Tested on Table 19.

In [8]:
import re

TABLE_HEADER = re.compile(r"^TABLE\s+(\d+)\s*:", re.MULTILINE)

def locate_table(text, table_number):
    headers = [(m.start(), int(m.group(1))) for m in TABLE_HEADER.finditer(text)]
    for i, (start, number) in enumerate(headers):
        if number != table_number:
            continue
        end = len(text)
        for next_start, next_number in headers[i + 1:]:
            if next_number != table_number:
                end = next_start
                break
        return text[start:end]
    return None

block = locate_table(full_text, 19)
print(block[:600])

TABLE 19: PER CAPITA NET STATE DOMESTIC PRODUCT
(Current Prices)
(₹)
Base: 2011-12
State/Union Territory
2011-12 2012-13 2013-14 2014-15 2015-16 2016-17
Andaman & Nicobar Islands 89,100 98,777 1,11,087 1,26,344 1,37,064 1,53,904
Andhra Pradesh 69,000 74,687 82,870 93,903 1,08,002 1,20,676
Arunachal Pradesh 73,540 82,626 94,135 1,14,789 1,16,985 1,24,129
Assam 41,142 44,599 49,734 52,895 60,817 66,330
Bihar 21,750 24,487 26,948 28,671 30,404 34,045
Chandigarh 1,58,967 1,80,457 2,03,356 2,12,594 2,30,009 2,52,236
Chhattisgarh 55,177 60,849 69,880 72,936 72,991 83,285
Delhi 1,85,001 2,05,568 2,27


### First table parser
A first version that turns a table block into rows of state, year and value, with small helpers to spot years and clean numbers.

In [9]:
import pandas as pd

YEAR = re.compile(r"^\d{4}-\d{2}$")
VALUE_TOKEN = re.compile(r"^\d[\d,]*\*?$")

def is_value(token):
    return bool(VALUE_TOKEN.match(token)) or token in {".", "-"}

def clean_number(token):
    token = token.strip().rstrip("*")
    if token in {".", "-", ""}:
        return None
    return float(token.replace(",", ""))

def parse_table(block):
    lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
    records = []
    years = None
    for line in lines:
        tokens = line.split()
        if tokens and all(YEAR.match(t) for t in tokens):
            years = tokens
            continue
        if years is None or line.upper().startswith("TABLE"):
            continue
        cut = next((i for i, t in enumerate(tokens) if is_value(t)), None)
        if cut is None or cut == 0:
            continue
        state = " ".join(tokens[:cut])
        values = [t for t in tokens[cut:] if is_value(t)]
        for year, value in zip(years, values):
            records.append((state, year, clean_number(value)))
    df = pd.DataFrame(records, columns=["state", "year", "value"])
    return df.drop_duplicates(["state", "year"]).reset_index(drop=True)

### Test the parser
Run it on Table 19 and check the shape and Bihar's rows.

In [10]:
df19 = parse_table(block)
print(df19.shape, "|", df19["state"].nunique(), "states")
print(sorted(df19["year"].unique()))
print(df19[df19["state"] == "Bihar"].to_string(index=False))

(470, 3) | 34 states
['2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
state    year   value
Bihar 2011-12 21750.0
Bihar 2012-13 24487.0
Bihar 2013-14 26948.0
Bihar 2014-15 28671.0
Bihar 2015-16 30404.0
Bihar 2016-17 34045.0
Bihar 2017-18 36850.0
Bihar 2018-19 40715.0
Bihar 2019-20 44175.0
Bihar 2020-21 42128.0
Bihar 2021-22 47296.0
Bihar 2022-23 54637.0
Bihar 2023-24 62201.0
Bihar 2024-25 69321.0


### Better year and number checks
Stronger patterns for financial years, calendar years and numbers.

In [11]:
YEAR_FY  = re.compile(r"^\d{4}-\d{2}$")
YEAR_CAL = re.compile(r"^\d{4}$")
NUMBER   = re.compile(r"^-?\d[\d,]*(\.\d+)?\*?$")

def is_year(token):
    return bool(YEAR_FY.match(token) or YEAR_CAL.match(token))

def is_value(token):
    return bool(NUMBER.match(token)) or token in {".", "-"}

def clean_number(token):
    token = token.strip().rstrip("*")
    if token in {".", "-", ""}:
        return None
    return float(token.replace(",", ""))

### Parser that reads the header row
Updated parser that picks up the year columns from the State/Union Territory header line.

In [12]:
def parse_table(block):
    if block is None:
        return pd.DataFrame(columns=["state", "year", "value"])
    lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
    records = []
    years = None
    for line in lines:
        tokens = line.split()
        low = line.lower()
        if "state/union territory" in low or "state/ut" in low:
            found = [t for t in tokens if is_year(t)]
            if found:
                years = found
            continue
        if tokens and all(YEAR_FY.match(t) for t in tokens):
            years = tokens
            continue
        if years is None or line.upper().startswith("TABLE"):
            continue
        cut = next((i for i, t in enumerate(tokens) if is_value(t)), None)
        if cut is None or cut == 0:
            continue
        state = " ".join(tokens[:cut])
        if state.isupper():          # skip region rows like NORTHERN REGION
            continue
        values = [t for t in tokens[cut:] if is_value(t)]
        for year, value in zip(years, values):
            records.append((state, year, clean_number(value)))
    df = pd.DataFrame(records, columns=["state", "year", "value"])
    return df.drop_duplicates(["state", "year"]).reset_index(drop=True)

### Try it on many tables
Run the parser across all the tables the index needs and print row and state counts.

In [13]:
def extract(table_number):
    return parse_table(locate_table(full_text, table_number))

for n, name in [(7,"Life expectancy"),(8,"Unemployment"),(138,"Per capita power"),
                (146,"Roads"),(148,"T&D losses"),(149,"Telephones"),
                (153,"Credit-deposit ratio"),(33,"Manufacturing GVA"),(25,"Total GVA"),
                (116,"Factories"),(130,"GFCF"),(1,"GER")]:
    df = extract(n)
    print(f"T{n:<4} {name:<22} rows={df.shape[0]:<5} states={df['state'].nunique()}")

T7    Life expectancy        rows=491   states=22
T8    Unemployment           rows=0     states=0
T138  Per capita power       rows=745   states=36
T146  Roads                  rows=576   states=36
T148  T&D losses             rows=683   states=37
T149  Telephones             rows=594   states=28
T153  Credit-deposit ratio   rows=814   states=38
T33   Manufacturing GVA      rows=470   states=34
T25   Total GVA              rows=470   states=34
T116  Factories              rows=0     states=0
T130  GFCF                   rows=0     states=0
T1    GER                    rows=0     states=0


### Look at the unemployment block
Print the raw text of Table 8 to see why it does not parse cleanly.

In [14]:
block8 = locate_table(full_text, 8)
print(block8[:700])

TABLE 8: STATE-WISE UNEMPLOYMENT RATE – USUAL STATUS (ADJUSTED)
(Rural Male)
(Per Thousands)
State/Union Territory 1993-941999-002004-052009-10 2011-122017-182018-192019-202020-212021-222022-232023-24
Andhra Pradesh 7 10 10 13 17 45 49 44 47 41 35 38
Arunachal Pradesh 16 8 11 15 17 43 61 54 35 65 45 54
Assam 46 32 24 34 43 74 62 68 30 31 12 36
Bihar 19 22 18 21 27 72 106 54 45 60 44 33
Chhattisgarh . . 8 9 11 27 26 32 27 19 19 16
Delhi - 39 20 18 94 36 6 25 63 43 107 36
Goa 70 69 91 35 66 107 24 57 76 107 87 71
Gujarat 13 6 8 8 4 55 38 20 10 19 19 5
Haryana 15 11 28 21 26 90 100 68 59 91 66 36
Himachal Pradesh 9 18 16 19 11 62 53 44 38 45 33 32
Jammu & Kashmir 9 12 17 18 22 37 29 35 35 24 24


### Parser adjusted for the header years
Another version that reads the years out of the header line with a regex.

In [15]:
def parse_table(block):
    if block is None:
        return pd.DataFrame(columns=["state", "year", "value"])
    lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
    records = []
    years = None
    for line in lines:
        tokens = line.split()
        low = line.lower()
        if "state/union territory" in low or "state/ut" in low:
            found = re.findall(r"\d{4}-\d{2}", line) or re.findall(r"\d{4}", line)
            if found:
                years = found
            continue
        if tokens and all(YEAR_FY.match(t) for t in tokens):
            years = tokens
            continue
        if years is None or line.upper().startswith("TABLE"):
            continue
        cut = next((i for i, t in enumerate(tokens) if is_value(t)), None)
        if cut is None or cut == 0:
            continue
        state = " ".join(tokens[:cut])
        if state.isupper():
            continue
        values = [t for t in tokens[cut:] if is_value(t)]
        for year, value in zip(years, values):
            records.append((state, year, clean_number(value)))
    df = pd.DataFrame(records, columns=["state", "year", "value"])
    return df.drop_duplicates(["state", "year"]).reset_index(drop=True)

### Re-run the batch check
Run the batch test again after the change.

In [16]:
for n, name in [(7,"Life expectancy"),(8,"Unemployment"),(138,"Per capita power"),
                (146,"Roads"),(148,"T&D losses"),(149,"Telephones"),
                (153,"Credit-deposit ratio"),(33,"Manufacturing GVA"),(25,"Total GVA"),
                (116,"Factories"),(130,"GFCF"),(1,"GER")]:
    df = extract(n)
    print(f"T{n:<4} {name:<22} rows={df.shape[0]:<5} states={df['state'].nunique()}")

T7    Life expectancy        rows=491   states=22
T8    Unemployment           rows=444   states=37
T138  Per capita power       rows=745   states=36
T146  Roads                  rows=576   states=36
T148  T&D losses             rows=683   states=37
T149  Telephones             rows=594   states=28
T153  Credit-deposit ratio   rows=814   states=38
T33   Manufacturing GVA      rows=470   states=34
T25   Total GVA              rows=470   states=34
T116  Factories              rows=0     states=0
T130  GFCF                   rows=0     states=0
T1    GER                    rows=0     states=0


## Factories table (printed sideways)
This table has states across the top instead of down the side, so the simple parser cannot read it. First, see what the built-in table reader returns.

In [17]:
target = "NUMBER OF FACTORIES"

with pdfplumber.open(pdf_path) as pdf:
    pages = [i for i, p in enumerate(pdf.pages) if target in (p.extract_text() or "")]
    print("pages with this table:", pages)
    first = pdf.pages[pages[0]].extract_tables()

print("tables found on first page:", len(first))
for table in first:
    for row in table[:8]:
        print(row)
    print("-----")

pages with this table: [268, 269]
tables found on first page: 0


### Try the built-in table reader
Try pdfplumber's table detection with a text-based strategy.

In [18]:
settings = {"vertical_strategy": "text", "horizontal_strategy": "text"}

with pdfplumber.open(pdf_path) as pdf:
    tables = pdf.pages[268].extract_tables(settings)

print("tables:", len(tables))
if tables:
    for row in tables[0][:10]:
        print(row)

tables: 1
['', 'Anda', 'man', '', '', '', '', '', '', '', '', '', '', 'D', '', 'adra &', '']
['', '', '', 'Andhr', 'a', 'Arunac', 'hal', '', '', '', '', '', '', '', '', '', 'Daman &']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['Year', '& Nic', 'obar', 'Prades', 'h', 'Prade', 'sh', 'Assam', '', 'Bihar', 'Cha', 'ndigarh', 'Ch', 'hattisgarh', '', 'Nagar', 'Diu']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', 'Isl', 'ands', '', '', '', '', '', '', '', '', '', '', '', '', 'Haveli', '']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['2004-05', '', '21', '15,57', '2', '', '-', '1,710', '', '1,674', '', '287', '', '1,343', '', '1,059', '1,681']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['2005-06', '', '12', '15,79', '0', '', '-', '1,864', '', '1,669', '', '296', '', '1,478', '', '1,124', '1,534']


### Read the words with positions
Pull each word with its x and y position and group them into lines. This is the base for coordinate-based parsing.

In [19]:
from collections import defaultdict

with pdfplumber.open(pdf_path) as pdf:
    words = pdf.pages[268].extract_words(use_text_flow=False)

lines = defaultdict(list)
for w in words:
    lines[round(w["top"])].append((round(w["x0"]), w["text"]))

for top in sorted(lines)[:25]:
    row = sorted(lines[top])
    print(top, [f"{x}:{t}" for x, t in row])

34 ['179:TABLE', '218:116:', '242:STATE-WISE', '312:NUMBER', '363:OF', '381:FACTORIES']
51 ['547:(Units)']
68 ['129:Andaman', '487:Dadra', '515:&']
73 ['187:Andhra', '227:Arunachal', '530:Daman', '563:&']
78 ['54:Year', '127:&', '136:Nicobar', '281:Assam', '329:Bihar', '362:Chandigarh', '421:Chhattisgarh', '496:Nagar']
84 ['183:Pradesh', '236:Pradesh', '555:Diu']
89 ['139:Islands', '495:Haveli']
104 ['54:2004-05', '160:21', '191:15,572', '268:-', '288:1,710', '330:1,674', '397:287', '454:1,343', '499:1,059', '547:1,681']
119 ['54:2005-06', '160:12', '191:15,790', '268:-', '288:1,864', '330:1,669', '397:296', '454:1,478', '499:1,124', '547:1,534']
134 ['54:2006-07', '160:12', '191:15,879', '268:-', '288:1,967', '330:1,602', '397:274', '454:1,779', '499:1,054', '547:1,523']
149 ['54:2007-08', '160:12', '191:16,741', '268:-', '288:1,859', '330:1,783', '397:294', '454:1,854', '499:1,014', '547:1,487']
164 ['54:2008-09', '160:12', '191:16,903', '268:-', '289:2,211', '330:1,775', '397:278', 

### Split header from body
split_header_body separates the heading band from the year rows using the first row that starts with a year.

In [20]:
import re

YEAR_RE = re.compile(r"^\d{4}-\d{2}$")

def split_header_body(words):
    """Split a page's words into the header band and the year-row body,
    using the first row that starts with a year."""
    year_tops = [w["top"] for w in words if YEAR_RE.match(w["text"])]
    if not year_tops:
        raise ValueError("No year row found: unexpected header geometry")
    body_top = min(year_tops)
    header = [w for w in words if w["top"] < body_top]
    body = [w for w in words if w["top"] >= body_top]
    print(f"[split] header words={len(header)} body words={len(body)} body starts at top={round(body_top)}")
    return header, body

### Run the split
Run it on page 268.

In [21]:
with pdfplumber.open(pdf_path) as pdf:
    words = pdf.pages[268].extract_words(use_text_flow=False)

header, body = split_header_body(words)

[split] header words=27 body words=393 body starts at top=104


### Find the value columns
find_value_columns groups the numbers by their right edge to work out where each column sits.

In [22]:
def find_value_columns(body, gap=20):
    """Find column positions from the body values using their right edges.
    Returns (centers, boundaries); boundaries are midpoints between centers."""
    edges = sorted(w["x1"] for w in body if is_value(w["text"]))
    if not edges:
        raise ValueError("No value words in body: unexpected header geometry")
    clusters = [[edges[0]]]
    for x in edges[1:]:
        if x - clusters[-1][-1] <= gap:
            clusters[-1].append(x)
        else:
            clusters.append([x])
    centers = [sum(c) / len(c) for c in clusters]
    boundaries = [(centers[i] + centers[i + 1]) / 2 for i in range(len(centers) - 1)]
    print(f"[columns] found {len(centers)} columns at x approx {[round(c) for c in centers]}")
    return centers, boundaries

### Run the column detection
Run it on the body.

In [23]:
centers, boundaries = find_value_columns(body)

[columns] found 9 columns at x approx [172, 224, 279, 311, 348, 406, 467, 517, 570]


### Verified state order (first draft)
The left-to-right state order read by eye from the printed header. This is kept as part of the specification, not guessed by code.

In [24]:
# Extraction specification (human-verified) - NOT parser logic.
# Verified left-to-right state order for each page block of a transposed table.
# Read directly from the printed table header and confirmed by eye.
TRANSPOSED_STATE_ORDER = {
    ("factories", 268): [
        "Andaman & Nicobar Islands", "Andhra Pradesh", "Arunachal Pradesh",
        "Assam", "Bihar", "Chandigarh", "Chhattisgarh",
        "Dadra & Nagar Haveli", "Daman & Diu",
    ],
    # remaining blocks (page 269, ...) added after we inspect their headers
}

### Match columns to states
assign_columns lines up the detected columns with the verified state names and fails if they do not match.

In [25]:
def assign_columns(centers, header, state_order):
    """Map detected value columns to verified state names from configuration.
    Raises ValueError if the configuration does not match the page geometry."""
    if len(state_order) != len(centers):
        raise ValueError(f"state count {len(state_order)} != column count {len(centers)}")

    anchors = []
    for state in state_order:
        first = state.split()[0]
        hits = [w for w in header if w["text"] == first]
        if not hits:
            raise ValueError(f"header word for '{state}' (token '{first}') not found")
        anchors.append((min(h["x0"] for h in hits), state))

    ordered = [s for _, s in sorted(anchors)]
    if ordered != list(state_order):
        raise ValueError(f"header order does not match configuration:\n  header={ordered}\n  config={state_order}")

    mapping = dict(zip(sorted(centers), state_order))
    print(f"[assign] {len(mapping)} columns named, count and order validated")
    return mapping

### Run the matching
Run it for page 268.

In [26]:
state_order = TRANSPOSED_STATE_ORDER[("factories", 268)]
mapping = assign_columns(centers, header, state_order)
mapping

[assign] 9 columns named, count and order validated


{171.5806000000003: 'Andaman & Nicobar Islands',
 224.27119000000025: 'Andhra Pradesh',
 279.0745075000003: 'Arunachal Pradesh',
 311.4600666666678: 'Assam',
 347.91996250000034: 'Bihar',
 405.99707500000034: 'Chandigarh',
 466.6255750000003: 'Chhattisgarh',
 517.3318225000003: 'Dadra & Nagar Haveli',
 569.6514825000003: 'Daman & Diu'}

### Read the values into rows
parse_transposed_body puts each year row's numbers into the right state column, keeps blanks as missing, and stops on a collision.

In [29]:
import bisect
from collections import defaultdict

def parse_transposed_body(body, boundaries, state_order):
    """Map each year-row's values into columns; return (state, year, value) records.
    Empty cells (no value, no collision) are recorded as missing and logged.
    A collision (two values in one column) raises ValueError."""
    rows = defaultdict(list)
    for w in body:
        rows[round(w["top"])].append(w)

    records = []
    n_cols = len(state_order)
    for top in sorted(rows):
        words = sorted(rows[top], key=lambda w: w["x0"])
        year_words = [w for w in words if YEAR_RE.match(w["text"])]
        if not year_words:
            continue
        year = year_words[0]["text"]
        slots = [None] * n_cols
        for w in words:
            if not is_value(w["text"]):
                continue
            idx = bisect.bisect_right(boundaries, w["x1"])
            if slots[idx] is not None:
                raise ValueError(f"row {year}: two values in column {idx} ({state_order[idx]})")
            slots[idx] = w["text"]
        empty = [state_order[i] for i, s in enumerate(slots) if s is None]
        if empty:
            print(f"[body] {year}: blank cell for {empty} -> recorded as missing")
        for state, token in zip(state_order, slots):
            value = clean_number(token) if token is not None else None
            records.append((state, year, value))
    print(f"[body] {len(records)} cells across {len(records) // n_cols} year-rows")
    return records

### Test on Bihar
Run it and check Bihar's values.

In [30]:
records = parse_transposed_body(body, boundaries, state_order)

import pandas as pd
dfp = pd.DataFrame(records, columns=["state", "year", "value"])
print(dfp[dfp["state"] == "Bihar"].to_string(index=False))

[body] 2023-24: blank cell for ['Daman & Diu'] -> recorded as missing
[body] 2004-05: blank cell for ['Assam'] -> recorded as missing
[body] 2005-06: blank cell for ['Assam'] -> recorded as missing
[body] 2006-07: blank cell for ['Assam'] -> recorded as missing
[body] 2007-08: blank cell for ['Assam'] -> recorded as missing
[body] 2008-09: blank cell for ['Assam'] -> recorded as missing
[body] 2009-10: blank cell for ['Assam'] -> recorded as missing
[body] 2010-11: blank cell for ['Assam'] -> recorded as missing
[body] 2011-12: blank cell for ['Assam'] -> recorded as missing
[body] 2012-13: blank cell for ['Assam'] -> recorded as missing
[body] 2013-14: blank cell for ['Assam'] -> recorded as missing
[body] 2014-15: blank cell for ['Assam'] -> recorded as missing
[body] 2015-16: blank cell for ['Assam'] -> recorded as missing
[body] 2016-17: blank cell for ['Assam'] -> recorded as missing
[body] 2017-18: blank cell for ['Assam'] -> recorded as missing
[body] 2018-19: blank cell for ['A

### Split a page into blocks
split_page_into_blocks handles a page that holds two stacked mini-tables by splitting them into separate header and body blocks.

In [33]:
def split_page_into_blocks(words):
    """Split one page into stacked blocks: each block is (header_words, body_words).
    Boundaries are real header bands (with letters) or year resets. Stray lines ignored."""
    lines = defaultdict(list)
    for w in words:
        lines[round(w["top"])].append(w)

    def classify(top):
        row = sorted(lines[top], key=lambda w: w["x0"])
        if row and YEAR_RE.match(row[0]["text"]):
            return "year", row[0]["text"]
        if any(re.search(r"[A-Za-z]", w["text"]) for w in row):
            return "header", None
        return "other", None

    blocks, header, body, prev_year = [], [], [], None
    for top in sorted(lines):
        kind, y = classify(top)
        if kind == "header":
            if body:
                blocks.append((header, body))
                header, body, prev_year = [], [], None
            header.extend(lines[top])
        elif kind == "year":
            if prev_year is not None and y <= prev_year:
                blocks.append((header, body))
                header, body, prev_year = [], [], None
            body.extend(lines[top])
            prev_year = y
        # "other" lines are ignored
    if body:
        blocks.append((header, body))

    print(f"[blocks] {len(blocks)} block(s) on this page")
    for i, (h, b) in enumerate(blocks):
        yrs = sorted({w['text'] for w in b if YEAR_RE.match(w['text'])})
        htext = " ".join(w["text"] for w in sorted(h, key=lambda w: (round(w["top"]), w["x0"])))
        print(f"  block {i}: {len(yrs)} years [{yrs[0]}..{yrs[-1]}]  header: {htext[:160]}")
    return blocks

### Run the block split
Run it on page 268.

In [34]:
with pdfplumber.open(pdf_path) as pdf:
    words = pdf.pages[268].extract_words(use_text_flow=False)

blocks = split_page_into_blocks(words)

[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: TABLE 116: STATE-WISE NUMBER OF FACTORIES (Units) Andaman Dadra & Andhra Arunachal Daman & Year & Nicobar Assam Bihar Chandigarh Chhattisgarh Nagar Pradesh Prad
  block 1: 20 years [2004-05..2023-24]  header: Himachal Jammu & Year Delhi Goa Gujarat Haryana Jharkhand Karnataka Pradesh Kashmir


### State order per block
The verified state order written out block by block.

In [35]:
TRANSPOSED_STATE_ORDER = {
    ("factories", 268): [
        [   # block 0
            "Andaman & Nicobar Islands", "Andhra Pradesh", "Arunachal Pradesh",
            "Assam", "Bihar", "Chandigarh", "Chhattisgarh",
            "Dadra & Nagar Haveli", "Daman & Diu",
        ],
        [   # block 1
            "Delhi", "Goa", "Gujarat", "Haryana", "Himachal Pradesh",
            "Jammu & Kashmir", "Jharkhand", "Karnataka",
        ],
    ],
    # ("factories", 269): filled after we read page 269 headers
}

### Split page 269
Run the block split on the second page of the table.

In [36]:
with pdfplumber.open(pdf_path) as pdf:
    words269 = pdf.pages[269].extract_words(use_text_flow=False)

blocks269 = split_page_into_blocks(words269)

[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: (Concld.) TABLE 116: STATE-WISE NUMBER OF FACTORIES (Units) Madhya Year Kerala Ladakh Maharashtra Manipur Meghalaya Mizoram Nagaland Odisha Puducherry Punjab Pr
  block 1: 20 years [2004-05..2023-24]  header: Tamil Uttar Uttara- West Lakshad- Year Rajasthan Sikkim Telangana Tripura All-India Nadu Pradesh khand Bengal weep


### Final state-order config
The full verified state order for both pages, with a note on why it is kept by hand.

In [37]:
# The RBI PDF layout does not allow reliable automatic reconstruction of state order
# because some wrapped and hyphenated headers are ambiguous. Therefore, the verified
# state order is treated as part of the extraction specification, while geometry is
# used only for column boundaries and value assignment.
TRANSPOSED_STATE_ORDER = {
    ("factories", 268): [
        ["Andaman & Nicobar Islands", "Andhra Pradesh", "Arunachal Pradesh",
         "Assam", "Bihar", "Chandigarh", "Chhattisgarh",
         "Dadra & Nagar Haveli", "Daman & Diu"],
        ["Delhi", "Goa", "Gujarat", "Haryana", "Himachal Pradesh",
         "Jammu & Kashmir", "Jharkhand", "Karnataka"],
    ],
    ("factories", 269): [
        ["Kerala", "Ladakh", "Madhya Pradesh", "Maharashtra", "Manipur",
         "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Puducherry", "Punjab"],
        ["Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura",
         "Uttar Pradesh", "Uttarakhand", "West Bengal", "Lakshadweep", "All-India"],
    ],
}

### Full extraction function
extract_transposed ties the pieces together to read a whole transposed table across pages.

In [38]:
def extract_transposed(pdf_path, table_name, pages, config):
    frames = []
    for page in pages:
        with pdfplumber.open(pdf_path) as pdf:
            words = pdf.pages[page].extract_words(use_text_flow=False)
        blocks = split_page_into_blocks(words)
        cfg = config[(table_name, page)]
        if len(blocks) != len(cfg):
            raise ValueError(f"page {page}: {len(blocks)} blocks detected, {len(cfg)} configured")
        for (header, body), state_order in zip(blocks, cfg):
            centers, boundaries = find_value_columns(body)
            if len(centers) != len(state_order):
                raise ValueError(
                    f"page {page}: {len(centers)} columns detected, {len(state_order)} configured")
            records = parse_transposed_body(body, boundaries, state_order)
            frames.append(pd.DataFrame(records, columns=["state", "year", "value"]))
    return pd.concat(frames, ignore_index=True)

### Extract the factories table
Run it for factories, drop the All-India row, and check the counts.

In [39]:
factories = extract_transposed(pdf_path, "factories", [268, 269], TRANSPOSED_STATE_ORDER)

factories = factories[factories["state"] != "All-India"].reset_index(drop=True)

n_states = factories["state"].nunique()
if n_states != 37:
    raise ValueError(f"expected 37 states/UTs after removing All-India, got {n_states}")
if factories.duplicated(["state", "year"]).any():
    raise ValueError("duplicate (state, year) pairs found")

print("factories:", factories.shape, "states:", n_states, "years:", factories["year"].nunique())

[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: TABLE 116: STATE-WISE NUMBER OF FACTORIES (Units) Andaman Dadra & Andhra Arunachal Daman & Year & Nicobar Assam Bihar Chandigarh Chhattisgarh Nagar Pradesh Prad
  block 1: 20 years [2004-05..2023-24]  header: Himachal Jammu & Year Delhi Goa Gujarat Haryana Jharkhand Karnataka Pradesh Kashmir
[columns] found 9 columns at x approx [170, 218, 271, 311, 352, 412, 477, 521, 569]
[body] 2023-24: blank cell for ['Daman & Diu'] -> recorded as missing
[body] 180 cells across 20 year-rows
[columns] found 8 columns at x approx [174, 230, 287, 343, 400, 457, 513, 570]
[body] 160 cells across 20 year-rows
[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: (Concld.) TABLE 116: STATE-WISE NUMBER OF FACTORIES (Units) Madhya Year Kerala Ladakh Maharashtra Manipur Meghalaya Mizoram Nagaland Odisha Puducherry Punjab Pr
  block 1: 20 years [2004-05..2023-24]  header: Tamil Uttar Uttara- West Laks

### Spot-check the numbers
Compare a few values against the printed table.

In [40]:
checks = [
    ("Bihar", "2004-05", 1674),
    ("Karnataka", "2004-05", 7596),
    ("Uttar Pradesh", "2004-05", 9582),
    ("Uttarakhand", "2004-05", 752),
]
for state, year, expected in checks:
    s = factories[(factories.state == state) & (factories.year == year)]["value"]
    got = s.iloc[0] if len(s) else None
    print(f"{'OK' if got == expected else 'MISMATCH'}: {state} {year} -> {got} (expected {expected})")

OK: Bihar 2004-05 -> 1674.0 (expected 1674)
OK: Karnataka 2004-05 -> 7596.0 (expected 7596)
OK: Uttar Pradesh 2004-05 -> 9582.0 (expected 9582)
OK: Uttarakhand 2004-05 -> 752.0 (expected 752)


### Save the factories table
Write it to data/processed.

In [41]:
out = Path("data/processed/factories_t116.csv")
factories.to_csv(out, index=False)
print("saved", out, factories.shape)

saved data\processed\factories_t116.csv (740, 3)


## Capital formation table (same layout)
GFCF is printed the same sideways way as factories, on different pages. Find those pages and inspect their blocks.

In [43]:
target = "GROSS FIXED CAPITAL FORMATION"
with pdfplumber.open(pdf_path) as pdf:
    gfcf_pages = [i for i, p in enumerate(pdf.pages) if target in (p.extract_text() or "")]
print("GFCF pages:", gfcf_pages)

for page in gfcf_pages:
    with pdfplumber.open(pdf_path) as pdf:
        words = pdf.pages[page].extract_words(use_text_flow=False)
    print("=== page", page, "===")
    split_page_into_blocks(words)

GFCF pages: [296, 297]
=== page 296 ===
[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: TABLE 130: STATE-WISE GROSS FIXED CAPITAL FORMATION Lakh) Andaman Dadra & Andhra Arunachal Daman & Year & Nicobar Assam Bihar Chandigarh Chhattisgarh Nagar Prad
  block 1: 20 years [2004-05..2023-24]  header: Himachal Jammu & Year Delhi Goa Gujarat Haryana Jharkhand Karnataka Pradesh Kashmir
=== page 297 ===
[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: (Concld.) TABLE 130: STATE-WISE GROSS FIXED CAPITAL FORMATION Lakh) Madhya Year Kerala Ladakh Maharashtra Manipur Meghalaya Mizoram Nagaland Odisha Puducherry P
  block 1: 20 years [2004-05..2023-24]  header: Tamil Uttar Uttara- West Lakshad- Year Rajasthan Sikkim Telangana Tripura All-India Nadu Pradesh khand Bengal weep


### Reuse the factories layout
The blocks and state order match factories, so reuse that config for the GFCF pages.

In [44]:
# GFCF (T130) has the same block structure and states as factories, on different pages.
TRANSPOSED_STATE_ORDER[("gfcf", 296)] = TRANSPOSED_STATE_ORDER[("factories", 268)]
TRANSPOSED_STATE_ORDER[("gfcf", 297)] = TRANSPOSED_STATE_ORDER[("factories", 269)]

### Extract and check GFCF
Run the extraction, drop All-India, validate the counts, and spot-check values.

In [45]:
gfcf = extract_transposed(pdf_path, "gfcf", [296, 297], TRANSPOSED_STATE_ORDER)
gfcf = gfcf[gfcf["state"] != "All-India"].reset_index(drop=True)

n = gfcf["state"].nunique()
if n != 37:
    raise ValueError(f"expected 37 states/UTs, got {n}")
if gfcf.duplicated(["state", "year"]).any():
    raise ValueError("duplicate (state, year) pairs found")
print("gfcf:", gfcf.shape, "states:", n, "years:", gfcf["year"].nunique())

checks = [
    ("Bihar", "2004-05", 11240),
    ("Karnataka", "2004-05", 607425),
    ("Uttar Pradesh", "2004-05", 513637),
    ("Uttarakhand", "2004-05", 77726),
]
for state, year, exp in checks:
    s = gfcf[(gfcf.state == state) & (gfcf.year == year)]["value"]
    got = s.iloc[0] if len(s) else None
    print(f"{'OK' if got == exp else 'MISMATCH'}: {state} {year} -> {got} (expected {exp})")

[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: TABLE 130: STATE-WISE GROSS FIXED CAPITAL FORMATION Lakh) Andaman Dadra & Andhra Arunachal Daman & Year & Nicobar Assam Bihar Chandigarh Chhattisgarh Nagar Prad
  block 1: 20 years [2004-05..2023-24]  header: Himachal Jammu & Year Delhi Goa Gujarat Haryana Jharkhand Karnataka Pradesh Kashmir
[columns] found 9 columns at x approx [162, 210, 263, 309, 355, 415, 480, 524, 572]
[body] 180 cells across 20 year-rows
[columns] found 8 columns at x approx [174, 230, 287, 343, 400, 457, 513, 570]
[body] 160 cells across 20 year-rows
[blocks] 2 block(s) on this page
  block 0: 20 years [2004-05..2023-24]  header: (Concld.) TABLE 130: STATE-WISE GROSS FIXED CAPITAL FORMATION Lakh) Madhya Year Kerala Ladakh Maharashtra Manipur Meghalaya Mizoram Nagaland Odisha Puducherry P
  block 1: 20 years [2004-05..2023-24]  header: Tamil Uttar Uttara- West Lakshad- Year Rajasthan Sikkim Telangana Tripura All-India Nadu Pradesh kh

### Save the GFCF table
Write it to data/processed.

In [46]:
gfcf.to_csv("data/processed/gfcf_t130.csv", index=False)
print("saved data/processed/gfcf_t130.csv", gfcf.shape)

saved data/processed/gfcf_t130.csv (740, 3)


## Enrolment table (GER)
Look at the Table 1 text block first.

In [47]:
block1 = locate_table(full_text, 1)
print(block1[:900])

TABLE 1: STATE-WISE GROSS ENROLMENT RATIO- 2022-23
Foundational* Preparatory Middle Secondary
State/Union
(Pre-Primary to Class II) (Class III to Class V) (Class VI to Class VIII) (Class IX to XII)
Territory
Male Female Total Male Female Total Male Female Total Male Female Total
Andaman and Nicobar
64.6 62.4 63.5 91.8 93.6 92.7 93.5 94.7 94.1 89.9 99.3 94.3
Islands
Andhra Pradesh 44.1 43.2 43.6 106.7 106.7 106.7 102.8 98.5 100.7 72.9 75.1 74.0
Arunachal Pradesh 64.8 65.7 65.3 100.1 103.3 101.6 82.2 88.8 85.5 58.3 62.9 60.6
Assam 52.2 54.0 53.1 106.6 112.5 109.4 87.3 97.0 92.1 51.3 60.8 55.9
Bihar 30.7 32.4 31.5 95.0 103.2 98.8 71.8 79.4 75.4 42.5 49.9 46.0
Chandigarh 64.7 65.7 65.2 111.3 120.1 115.2 114.7 127.8 120.4 108.6 124.9 115.5
Chhattisgarh 42.1 41.9 42.0 91.5 92.2 91.9 90.7 91.8 91.2 63.5 72.8 68.1
Dadra & Nagar Haveli
52.2 50.4 51.4 108.1 110.4 109.1 108.4 113.1 110.6 69.0 95.9 


### Find the GER pages and layout
Locate the pages and print the word positions of the header.

In [48]:
from collections import defaultdict

target = "GROSS ENROLMENT RATIO"
with pdfplumber.open(pdf_path) as pdf:
    ger_pages = [i for i, p in enumerate(pdf.pages) if target in (p.extract_text() or "")]
print("GER pages:", ger_pages)

with pdfplumber.open(pdf_path) as pdf:
    words = pdf.pages[ger_pages[0]].extract_words(use_text_flow=False)

lines = defaultdict(list)
for w in words:
    lines[round(w["top"])].append((round(w["x0"]), w["text"]))
for top in sorted(lines)[:16]:
    print(top, [f"{x}:{t}" for x, t in sorted(lines[top])])

GER pages: [16, 17, 18, 47, 48, 49, 50, 51, 52, 53]
32 ['153:TABLE', '192:1:', '205:STATE-WISE', '274:GROSS', '317:ENROLMENT', '390:RATIO-', '430:2022-23']
46 ['173:Foundational*', '282:Preparatory', '401:Middle', '500:Secondary']
53 ['74:State/Union']
57 ['151:(Pre-Primary', '205:to', '216:Class', '241:II)', '265:(Class', '294:III', '304:to', '315:Class', '342:V)', '368:(Class', '398:VI', '409:to', '420:Class', '446:VIII)', '490:(Class', '520:IX', '531:to', '542:XII)']
64 ['81:Territory']
70 ['156:Male', '184:Female', '223:Total', '264:Male', '292:Female', '331:Total', '371:Male', '400:Female', '439:Total', '479:Male', '507:Female', '547:Total']
84 ['52:Andaman', '93:and', '111:Nicobar']
89 ['164:64.6', '198:62.4', '232:63.5', '272:91.8', '306:93.6', '340:92.7', '379:93.5', '413:94.7', '447:94.1', '487:89.9', '521:99.3', '555:94.3']
94 ['52:Islands']
109 ['52:Andhra', '84:Pradesh', '164:44.1', '198:43.2', '232:43.6', '267:106.7', '301:106.7', '335:106.7', '374:102.8', '413:98.5', '442

### Read fixed-width value rows
extract_value_rows is a general reader for tables with fixed number columns, keeping blanks as missing.

In [49]:
def extract_value_rows(page_words, expected_columns, row_filter=None):
    """Generic: return ordered value rows for any table with fixed-width numeric
    columns. Column boundaries are detected from the page's value words; the detected
    count must equal expected_columns. Each row is (top, [value_or_None, ...]) with
    blanks kept as None. row_filter(top, values) can drop non-data rows."""
    centers, boundaries = find_value_columns(page_words)
    if len(centers) != expected_columns:
        raise ValueError(f"detected {len(centers)} columns, expected {expected_columns}")

    lines = defaultdict(list)
    for w in page_words:
        if is_value(w["text"]):
            lines[round(w["top"])].append(w)

    rows = []
    for top in sorted(lines):
        slots = [None] * expected_columns
        for w in lines[top]:
            idx = bisect.bisect_right(boundaries, w["x1"])
            if 0 <= idx < expected_columns and slots[idx] is None:
                slots[idx] = clean_number(w["text"])
        if any(s is not None for s in slots):
            if row_filter is None or row_filter(top, slots):
                rows.append((top, slots))
    return rows

### Test on one page
Run it on page 16 and check the row count.

In [50]:
with pdfplumber.open(pdf_path) as pdf:
    words16 = pdf.pages[16].extract_words(use_text_flow=False)

rows = extract_value_rows(words16, expected_columns=12)
print("value rows:", len(rows))
for top, vals in rows[:2]:
    print(top, vals)

[columns] found 12 columns at x approx [182, 216, 250, 289, 323, 357, 397, 431, 465, 505, 539, 573]
value rows: 38
89 [64.6, 62.4, 63.5, 91.8, 93.6, 92.7, 93.5, 94.7, 94.1, 89.9, 99.3, 94.3]
109 [44.1, 43.2, 43.6, 106.7, 106.7, 106.7, 102.8, 98.5, 100.7, 72.9, 75.1, 74.0]


### Print every row
List all rows to eyeball the first and last values.

In [51]:
for i, (top, vals) in enumerate(rows):
    print(i, top, vals[0], vals[-1])

0 89 64.6 94.3
1 109 44.1 74.0
2 127 64.8 60.6
3 144 52.2 55.9
4 162 30.7 46.0
5 179 64.7 115.5
6 197 42.1 68.1
7 217 52.2 80.0
8 237 49.2 97.3
9 254 74.8 102.1
10 272 35.4 61.9
11 289 38.2 81.7
12 307 61.7 93.2
13 324 87.5 56.2
14 342 41.0 49.5
15 359 48.1 80.8
16 377 52.8 91.8
17 394 87.8 73.0
18 412 60.8 71.8
19 429 35.9 57.9
20 446 40.3 78.9
21 464 81.8 70.8
22 481 125.0 62.7
23 499 98.2 73.7
24 516 76.5 47.8
25 534 35.1 66.8
26 551 65.7 97.6
27 569 74.1 85.6
28 586 48.9 72.9
29 603 89.3 71.7
30 621 53.3 87.2
31 638 50.6 83.7
32 656 52.3 67.8
33 673 31.4 58.9
34 691 50.9 85.5
35 708 53.0 81.2
36 726 41.3 67.6
37 754 None None


### GER settings
The pages, columns, level and gender labels, and state order for the enrolment table.

In [52]:
GER_CONFIG = {
    "pages": {16: "2022-23", 17: "2023-24", 18: "2024-25"},
    "expected_columns": 12,
    "column_labels": [
        ("Foundational", "Male"), ("Foundational", "Female"), ("Foundational", "Total"),
        ("Preparatory", "Male"), ("Preparatory", "Female"), ("Preparatory", "Total"),
        ("Middle", "Male"), ("Middle", "Female"), ("Middle", "Total"),
        ("Secondary", "Male"), ("Secondary", "Female"), ("Secondary", "Total"),
    ],
    "state_order": [
        "Andaman & Nicobar Islands", "Andhra Pradesh", "Arunachal Pradesh", "Assam",
        "Bihar", "Chandigarh", "Chhattisgarh", "Dadra & Nagar Haveli and Daman & Diu",
        "Delhi", "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jammu & Kashmir",
        "Jharkhand", "Karnataka", "Kerala", "Ladakh", "Lakshadweep", "Madhya Pradesh",
        "Maharashtra", "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Odisha",
        "Puducherry", "Punjab", "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana",
        "Tripura", "Uttar Pradesh", "Uttarakhand", "West Bengal", "All India",
    ],
}

### Build the GER table
extract_ger reads all three years, drops footnote rows, and labels each value by level and gender.

In [53]:
def extract_ger(pdf_path, config):
    labels, states, ncol = config["column_labels"], config["state_order"], config["expected_columns"]
    keep = lambda top, vals: sum(v is not None for v in vals) >= 8   # drop footnote/phantom rows
    records = []
    for page, year in config["pages"].items():
        with pdfplumber.open(pdf_path) as pdf:
            words = pdf.pages[page].extract_words(use_text_flow=False)
        rows = extract_value_rows(words, ncol, row_filter=keep)
        if len(rows) != len(states):
            raise ValueError(f"page {page} ({year}): {len(rows)} rows, expected {len(states)}")
        for (top, vals), state in zip(rows, states):
            for (level, gender), value in zip(labels, vals):
                records.append((state, year, level, gender, value))
    return pd.DataFrame(records, columns=["state", "year", "level", "gender", "ger"])

### Extract and check GER
Run it, drop the All-India row, validate, and spot-check a few states.

In [54]:
ger = extract_ger(pdf_path, GER_CONFIG)
ger = ger[ger["state"] != "All India"].reset_index(drop=True)

n = ger["state"].nunique()
if n != 36:
    raise ValueError(f"expected 36 states/UTs, got {n}")
if ger.duplicated(["state", "year", "level", "gender"]).any():
    raise ValueError("duplicate rows")
print("ger:", ger.shape, "states:", n, "years:", ger["year"].nunique())

checks = [
    ("Bihar", 46.0), ("Uttar Pradesh", 58.9), ("Uttarakhand", 85.5), ("Lakshadweep", 71.8),
]
for state, exp in checks:
    s = ger[(ger.state == state) & (ger.year == "2022-23")
            & (ger.level == "Secondary") & (ger.gender == "Total")]["ger"]
    got = s.iloc[0] if len(s) else None
    print(f"{'OK' if got == exp else 'MISMATCH'}: {state} Secondary Total 2022-23 -> {got} (exp {exp})")

ger.to_csv("data/processed/ger_t1.csv", index=False)
print("saved data/processed/ger_t1.csv", ger.shape)

[columns] found 12 columns at x approx [182, 216, 250, 289, 323, 357, 397, 431, 465, 505, 539, 573]
[columns] found 12 columns at x approx [182, 216, 250, 289, 323, 357, 397, 431, 465, 505, 539, 573]
[columns] found 12 columns at x approx [179, 213, 247, 287, 321, 355, 394, 429, 463, 502, 536, 570]
ger: (1296, 5) states: 36 years: 3
OK: Bihar Secondary Total 2022-23 -> 46.0 (exp 46.0)
OK: Uttar Pradesh Secondary Total 2022-23 -> 58.9 (exp 58.9)
OK: Uttarakhand Secondary Total 2022-23 -> 85.5 (exp 85.5)
OK: Lakshadweep Secondary Total 2022-23 -> 71.8 (exp 71.8)
saved data/processed/ger_t1.csv (1296, 5)
